In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [2]:
X, y = make_classification(
    n_samples=10000,     # 10,000 customers
    n_features=15,       # 15 input features
    n_informative=10,    # 10 features are actually useful
    n_redundant=3,       # 3 redundant (noise) features
    random_state=42
)

In [3]:
feature_names = [
    'age', 'income', 'credit_score', 'loan_amount',
    'employment_years', 'num_accounts', 'debt_ratio',
    'payment_history', 'num_late_payments', 'credit_util',
    'num_inquiries', 'account_age', 'savings_balance',
    'monthly_expenses', 'collateral_value'
]

In [4]:
df = pd.DataFrame(X, columns=feature_names)
df['default'] = y 

In [5]:
df.head()

,age,income,credit_score,loan_amount,employment_years,num_accounts,debt_ratio,payment_history,num_late_payments,credit_util,num_inquiries,account_age,savings_balance,monthly_expenses,collateral_value,default
0,1.046080,1.599332,2.532688,-1.976311,0.687601,1.412669,1.522230,-0.899900,1.261921,-3.192460,0.344328,0.798130,-1.354848,2.396030,1.901351,1
1,0.359778,-0.861577,1.638884,0.445764,2.144193,-0.437632,-0.388196,0.724696,0.069161,0.874682,-0.888568,0.485751,0.327214,2.469600,-3.933467,0
2,1.516452,-4.385902,-0.535998,-1.017093,0.496900,4.593960,4.847821,2.065728,-0.840693,-0.354530,1.020490,-0.188313,-2.981050,3.154648,-1.948623,1
3,-0.269378,1.784872,-3.738618,0.437498,-7.610879,8.266258,1.485765,-0.830049,-0.145108,0.423583,-0.040116,4.617513,-2.550867,1.955587,2.813492,1
4,0.310363,-1.283132,0.903829,0.551854,0.666400,2.091127,2.909605,2.686448,1.191811,-2.558112,-1.050794,1.467806,-2.298239,4.178483,-1.573498,0


In [6]:
df.shape

(10000, 16)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_names], df['default'],
    test_size=0.2,           # 80% train, 20% test
    random_state=42,
    stratify=df['default']   # keep class distribution equal in both splits
)

In [8]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 15), (2000, 15), (8000,), (2000,))

In [9]:
base_model = lgb.LGBMClassifier(
    n_estimators=200,    # build 200 trees
    random_state=42,
    verbose=-1           # suppress training logs
)

In [10]:
base_model.fit(X_train, y_train)

,n_estimators,200
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [11]:
base_preds = base_model.predict(X_test)
base_proba = base_model.predict_proba(X_test)[:, 1]

In [12]:
print("\n--- Base Model Performance ---")
print(f"ROC-AUC : {roc_auc_score(y_test, base_proba):.4f}")
print(classification_report(y_test, base_preds))


--- Base Model Performance ---
ROC-AUC : 0.9815
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1000
           1       0.94      0.94      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000



In [13]:
param_dist = {
    # Number of boosting trees to build
    'n_estimators': [100, 200, 300, 500],

    # Maximum tree depth (-1 means no limit, leaf-wise controls complexity)
    'max_depth': [-1, 5, 7, 10, 15],

    # Maximum number of leaves per tree (key LightGBM parameter)
    'num_leaves': [20, 31, 50, 70, 100],

    # Learning rate — how much each tree contributes to final prediction
    'learning_rate': [0.01, 0.05, 0.1, 0.2],

    # Fraction of features used per tree (controls overfitting)
    'feature_fraction': [0.6, 0.7, 0.8, 0.9, 1.0],

    # Fraction of training data used per tree (bagging)
    'bagging_fraction': [0.6, 0.7, 0.8, 0.9, 1.0],

    # Frequency of bagging (0 = disabled)
    'bagging_freq': [0, 3, 5, 7],

    # L1 regularization — encourages sparse feature weights
    'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0],

    # L2 regularization — prevents large weights, reduces overfitting
    'reg_lambda': [0, 0.01, 0.1, 0.5, 1.0],

    # Minimum number of data points required in a leaf node
    'min_child_samples': [10, 20, 30, 50],
}

In [14]:
lgbm_estimator = lgb.LGBMClassifier(
    random_state=42,
    verbose=-1
)

In [15]:
random_search = RandomizedSearchCV(
    estimator=lgbm_estimator,
    param_distributions=param_dist,
    n_iter=30,           # try 30 random combinations (not all)
    cv=5,                # 5-fold cross validation
    scoring='roc_auc',   # optimize for ROC-AUC score
    n_jobs=-1,           # use all available CPU cores
    random_state=42,
    verbose=1            # show progress
)

In [16]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMClassifie...2, verbose=-1)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'bagging_fraction': [0.6, 0.7, ...], 'bagging_freq': [0, 3, ...], 'feature_fraction': [0.6, 0.7, ...], 'learning_rate': [0.01, 0.05, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a fu

In [17]:
best_model = random_search.best_estimator_

In [18]:
random_search.best_params_

{'reg_lambda': 0,
 'reg_alpha': 0.01,
 'num_leaves': 70,
 'n_estimators': 500,
 'min_child_samples': 10,
 'max_depth': -1,
 'learning_rate': 0.05,
 'feature_fraction': 1.0,
 'bagging_freq': 5,
 'bagging_fraction': 0.7}

In [19]:
tuned_preds = best_model.predict(X_test)
tuned_proba = best_model.predict_proba(X_test)[:, 1]

In [20]:
print(f"ROC-AUC : {roc_auc_score(y_test, tuned_proba):.4f}")
print(classification_report(y_test, tuned_preds))

ROC-AUC : 0.9832
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1000
           1       0.94      0.94      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000



In [21]:
importance_df = pd.DataFrame({
    'feature' : feature_names,
    'importance' : best_model.feature_importances_
}).sort_values("importance", ascending = False).reset_index(drop = True)

print("\nTop 5 Most Important Features:")
print(importance_df.head(5).to_string(index=False))


Top 5 Most Important Features:
          feature  importance
       debt_ratio        2957
 employment_years        2860
num_late_payments        2812
    num_inquiries        2697
           income        2686


## Cat boosting

In [22]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

In [23]:

np.random.seed(42)       # fix random seed so same data generates every run
n = 1000                 # total number of employee records to generate

# ── 5 Numerical Features ──────────────────

age = np.random.randint(22, 60, n)
# employee age between 22 and 60 years

monthly_salary = np.random.randint(20000, 120000, n)
# monthly salary in rupees between 20k and 1.2 lakh

years_at_company = np.random.randint(0, 20, n)
# how many years employee has been with the company

work_hours_per_week = np.random.randint(35, 70, n)
# weekly working hours — high = more stress = more attrition risk

performance_score = np.random.uniform(1.0, 5.0, n).round(1)
# performance rating given by manager — scale 1.0 to 5.0


# ── 3 Categorical Features ────────────────

department = np.random.choice(
    ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'],
    size=n
)
# which department the employee belongs to

job_level = np.random.choice(
    ['Junior', 'Mid', 'Senior', 'Lead', 'Manager'],
    size=n
)
# seniority level of the employee

work_mode = np.random.choice(
    ['Remote', 'Hybrid', 'On-site'],
    size=n,
    p=[0.3, 0.4, 0.3]    # 30% remote, 40% hybrid, 30% on-site (realistic distribution)
)
# where the employee works from


# ── Target Variable (Attrition) ───────────

# Create realistic attrition logic — not fully random
# Higher attrition chances when: low salary, high hours, low performance
attrition_prob = (
    0.1                                              # base probability 10%
    + (work_hours_per_week > 55) * 0.15             # +15% if overworked
    + (monthly_salary < 40000) * 0.20               # +20% if underpaid
    + (performance_score < 2.5) * 0.10              # +10% if low performer
    + (years_at_company < 2) * 0.10                 # +10% if new joiner
    + (work_mode == 'On-site') * 0.05               # +5% if forced on-site
)
attrition_prob = np.clip(attrition_prob, 0, 1)      # keep probability between 0 and 1

# Generate final binary target using the above probabilities
attrition = np.random.binomial(1, attrition_prob)   # 1 = will leave, 0 = will stay


# ── Assemble Full DataFrame ───────────────

df = pd.DataFrame({
    'age'                  : age,
    'monthly_salary'       : monthly_salary,
    'years_at_company'     : years_at_company,
    'work_hours_per_week'  : work_hours_per_week,
    'performance_score'    : performance_score,
    'department'           : department,        # categorical
    'job_level'            : job_level,         # categorical
    'work_mode'            : work_mode,         # categorical
    'attrition'            : attrition          # target column
})

In [24]:
df.head()

,age,monthly_salary,years_at_company,work_hours_per_week,performance_score,department,job_level,work_mode,attrition
0,50,22717,18,57,4.1,Engineering,Lead,Remote,0
1,36,79676,14,54,4.8,HR,Lead,On-site,1
2,29,47952,3,52,4.1,Sales,Mid,Remote,0
3,42,106444,12,52,3.4,HR,Mid,Remote,0
4,40,62460,18,36,4.4,Finance,Junior,On-site,0


In [25]:
df.shape

(1000, 9)

In [26]:
df.isnull().sum()

age                    0
monthly_salary         0
years_at_company       0
work_hours_per_week    0
performance_score      0
department             0
job_level              0
work_mode              0
attrition              0
dtype: int64

In [27]:
feature_cols = [
    'age', 'monthly_salary', 'years_at_company',
    'work_hours_per_week', 'performance_score',
    'department', 'job_level', 'work_mode'
]                             # all 8 feature columns (5 numerical + 3 categorical)

X = df[feature_cols]          # feature matrix — all input columns
y = df['attrition']           # target vector — what we want to predict

In [28]:
categorical_features = ['department', 'job_level', 'work_mode']   # names of cat columns

In [29]:
cat_feature_indices = [
    feature_cols.index(col)          # get position number of each categorical column
    for col in categorical_features
]

In [30]:
cat_feature_indices

[5, 6, 7]

In [31]:
for name, idx in zip(categorical_features, cat_feature_indices):
    print(f"  Column: {name:<20} Index: {idx}")   # show name and its index position

  Column: department           Index: 5
  Column: job_level            Index: 6
  Column: work_mode            Index: 7


In [32]:
for col in categorical_features:
    print(f"  {col}: {df[col].unique().tolist()}")  # show all unique category values

  department: ['Engineering', 'HR', 'Sales', 'Finance', 'Marketing']
  job_level: ['Lead', 'Mid', 'Junior', 'Senior', 'Manager']
  work_mode: ['Remote', 'On-site', 'Hybrid']


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,                    # feature matrix
    y,                    # target vector
    test_size=0.2,        # keep 20% data for testing
    random_state=42,      # fixed seed for reproducible split
    stratify=y            # maintain same attrition ratio in both train and test
)

In [34]:
model = CatBoostClassifier(
    iterations=300,           # build 300 decision trees sequentially
    learning_rate=0.05,       # each tree contributes 5% to the final prediction
    depth=6,                  # maximum depth of each tree — controls complexity
    l2_leaf_reg=5,            # L2 regularization to prevent overfitting
    random_seed=42,           # fixed seed for consistent results
    eval_metric='AUC',        # monitor AUC score during training
    loss_function='Logloss',  # log loss for binary classification
    early_stopping_rounds=30, # stop if no AUC improvement for 30 consecutive rounds
    verbose=50                # print training log every 50 iterations
)

In [37]:
model.fit(
    X_train,                           # training features
    y_train,                           # training labels
    cat_features=cat_feature_indices,  # tell CatBoost which columns are categorical
    eval_set=(X_test, y_test),         # validation set to monitor during training
    plot=False                         # dont open browser plot (keep output clean)
)

0:	test: 0.5355556	best: 0.5355556 (0)	total: 24.6ms	remaining: 7.36s
50:	test: 0.6192115	best: 0.6278136 (22)	total: 2.44s	remaining: 11.9s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.6278136201
bestIteration = 22

Shrink model to first 23 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=30, eval_metric='AUC', iterations=300, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

In [38]:
y_pred       = model.predict(X_test)             # predicted class labels (0 or 1)
y_pred_proba = model.predict_proba(X_test)[:, 1] # probability of attrition (class 1)

In [39]:
acc = accuracy_score(y_test, y_pred)             # fraction of correct predictions
auc = roc_auc_score(y_test, y_pred_proba)        # AUC — main metric for imbalanced data
print(classification_report(y_test, y_pred,
      target_names=['No Attrition', 'Attrition']))      # readable class names

              precision    recall  f1-score   support

No Attrition       0.78      1.00      0.87       155
   Attrition       0.00      0.00      0.00        45

    accuracy                           0.78       200
   macro avg       0.39      0.50      0.44       200
weighted avg       0.60      0.78      0.68       200



In [40]:
df["attrition"].value_counts()

attrition
0    775
1    225
Name: count, dtype: int64